In [2]:
"""
Setup — must run before the corpus audit cell.
"""
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Project root resolution (same pattern as your other notebooks)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment (.env at project root)
load_dotenv(PROJECT_ROOT / ".env", override=True)
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY not loaded — check .env"

# Build the retriever
from src.retrieval import HybridRetriever

CHROMA_PATH = str(PROJECT_ROOT / "chroma_db")    # adjust if your DB is elsewhere
COLLECTION_NAME = "financial_risk_10k"           # adjust if your collection name differs

retriever = HybridRetriever(
    chroma_path=CHROMA_PATH,
    collection_name=COLLECTION_NAME,
)
print("✓ Retriever ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading chunks from collection 'financial_risk_10k'...
Building BM25 index over 1401 chunks...
✓ HybridRetriever ready: 1401 chunks
✓ Retriever ready


In [3]:
"""
Corpus audit — what's actually retrievable from our chunks?

We can't write good test questions without knowing what topics 
the corpus actually covers.
"""
from collections import Counter

# 1. Confirm which tickers we have
all_tickers = [m.get("ticker") for m in retriever.chunk_metas]
ticker_counts = Counter(all_tickers)
print("Chunks per ticker:")
for t, n in sorted(ticker_counts.items()):
    print(f"  {t}: {n} chunks")

# 2. Probe for topics likely to appear in Risk Factors sections
topics_to_check = [
    # Cybersecurity cluster
    "cyber", "data breach", "ransomware", "information security",
    # Regulatory cluster
    "Basel", "Dodd-Frank", "CCAR", "stress test", "capital requirements",
    # Credit/market risk cluster
    "credit risk", "interest rate", "commercial real estate", "consumer credit",
    # Operational cluster
    "third party", "vendor", "outsourc", "geopolitical",
    # Specific (likely to fail — confirms they're not in Item 1A)
    "LIBOR", "SOFR", "stablecoin", "cryptocurrency", "climate",
]

print("\nTopic coverage across corpus:")
print(f"{'Topic':<28} {'Total':<8} {'Tickers covering it'}")
for topic in topics_to_check:
    hits = [
        meta.get("ticker") 
        for meta, text in zip(retriever.chunk_metas, retriever.chunk_texts)
        if topic.lower() in text.lower()
    ]
    unique_tickers = sorted(set(hits))
    print(f"{topic:<28} {len(hits):<8} {unique_tickers}")

Chunks per ticker:
  AXP: 142 chunks
  BAC: 166 chunks
  BLK: 143 chunks
  COF: 280 chunks
  GS: 170 chunks
  JPM: 125 chunks
  MA: 101 chunks
  PYPL: 174 chunks
  V: 100 chunks

Topic coverage across corpus:
Topic                        Total    Tickers covering it
cyber                        169      ['AXP', 'BAC', 'BLK', 'COF', 'GS', 'JPM', 'MA', 'PYPL', 'V']
data breach                  7        ['AXP', 'BAC', 'BLK', 'GS', 'JPM', 'MA']
ransomware                   13       ['AXP', 'BAC', 'BLK', 'COF', 'JPM', 'MA', 'PYPL', 'V']
information security         46       ['AXP', 'BAC', 'BLK', 'COF', 'GS', 'JPM', 'MA', 'PYPL']
Basel                        16       ['AXP', 'COF', 'GS', 'JPM']
Dodd-Frank                   14       ['BLK', 'COF', 'GS', 'JPM', 'V']
CCAR                         4        ['BAC', 'JPM']
stress test                  20       ['AXP', 'BAC', 'COF', 'JPM']
capital requirements         29       ['AXP', 'BAC', 'BLK', 'COF', 'GS', 'JPM', 'PYPL']
credit risk            

COF (Capital One) has 280 chunks — almost 2x the median. Either it has a much longer Item 1A, or your chunking produced more granular splits. Worth checking; might be why it dominated the LIBOR/SOFR result. Could be a retrieval bias to watch for.
MA and V have only ~100 chunks each. Payment networks have shorter Risk Factors sections than banks, which makes sense — they don't take credit risk the same way. They appear in fewer topical clusters (no Basel, no Dodd-Frank).

In [ ]:
# Read the actual chunks for a question to sanity-check its reference answer
response = pipeline.query("How do JPM and GS describe their stress-testing and capital requirements differently?")
for i, chunk in enumerate(response.retrieved_chunks, 1):
    print(f"[Chunk {i}] {chunk.metadata.get('ticker')}")
    print(chunk.text[:500])
    print()

NameError: name 'pipeline' is not defined

: 

In [ ]:
"""
Spot-check helper: pull chunks for a question and inspect them.

Use this to verify that my reference answer actually reflects 
what's in the corpus for that question.
"""
from src.generation import Generator
from src.pipeline import RAGPipeline

# Reuse the retriever you already built
generator = Generator(prompt_version="v2_enforced_shape")
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)


def inspect_question(question: str, top_k: int = 5, show_full: bool = False):
    """Show the chunks that would be retrieved for this question."""
    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    
    chunks = retriever.hybrid_search(question, top_k=top_k, candidate_pool=25)
    
    for i, chunk in enumerate(chunks, 1):
        ticker = chunk.metadata.get("ticker", "?")
        print(f"\n[Chunk {i}] Ticker: {ticker}  |  RRF score: {chunk.score:.4f}")
        print(f"          Dense rank: {chunk.dense_rank}  |  BM25 rank: {chunk.bm25_rank}")
        print("-" * 60)
        if show_full:
            print(chunk.text)
        else:
            # Show first 400 chars + last 200 chars
            text = chunk.text
            if len(text) > 600:
                print(text[:400])
                print("    [...]")
                print(text[-200:])
            else:
                print(text)
    print()


# Try it on one comparison question
inspect_question(
    "How do JPM and GS describe their stress-testing and capital requirements differently?"
)